# @classmethod and @staticmethod

Python has three kinds of methods inside a class:
- **Instance method** — receives `self` (the instance)
- **@classmethod** — receives `cls` (the class itself), not the instance
- **@staticmethod** — receives neither; just a plain function namespaced inside the class

## 1. Instance Method vs @classmethod vs @staticmethod

In [ ]:
class Temperature:
    unit = 'Celsius'  # class attribute shared by all instances

    def __init__(self, value: float):
        self.value = value  # instance attribute — unique to each object

    # ── Instance method ───────────────────────────────────────────────────────
    # Receives 'self' — can access and modify instance attributes
    def describe(self) -> str:
        return f'{self.value}° {Temperature.unit}'

    # ── @classmethod ──────────────────────────────────────────────────────────
    # Receives 'cls' (the class) instead of 'self' (the instance)
    # Common use: alternative constructors — create an object from different input formats
    @classmethod
    def from_fahrenheit(cls, f: float) -> 'Temperature':
        """Create a Temperature from a Fahrenheit value."""
        celsius = (f - 32) * 5 / 9  # convert first
        return cls(celsius)          # cls(...) is the same as Temperature(...)
                                     # using cls means subclasses work correctly too

    @classmethod
    def from_kelvin(cls, k: float) -> 'Temperature':
        """Create a Temperature from a Kelvin value."""
        return cls(k - 273.15)

    # ── @staticmethod ─────────────────────────────────────────────────────────
    # No 'self' or 'cls' — just a regular function that lives inside the class
    # Use when the logic is related to the class but doesn't need any class/instance data
    @staticmethod
    def celsius_to_fahrenheit(c: float) -> float:
        """Convert Celsius to Fahrenheit — doesn't need self or cls."""
        return c * 9 / 5 + 32


# Instance method — called on an object
t1 = Temperature(100)
print(t1.describe())                        # 100° Celsius

# @classmethod — called on the class, returns a new instance
t2 = Temperature.from_fahrenheit(212)       # alternative constructor
print(t2.describe())                        # 100.0° Celsius

t3 = Temperature.from_kelvin(373.15)
print(t3.describe())                        # 100.0° Celsius

# @staticmethod — called on the class, no instance needed
print(Temperature.celsius_to_fahrenheit(0)) # 32.0

## 2. @classmethod as Alternative Constructor (common pattern)

In [ ]:
from dataclasses import dataclass

@dataclass
class Contact:
    name:  str
    email: str
    phone: str

    def to_dict(self) -> dict:
        """Serialize to a plain dict (e.g. for JSON storage)."""
        return {'name': self.name, 'email': self.email, 'phone': self.phone}

    @classmethod
    def from_dict(cls, data: dict) -> 'Contact':
        """
        Alternative constructor — build a Contact from a dict.
        This is the @classmethod pattern: receive raw data, return a new instance.
        """
        return cls(
            name=data['name'],
            email=data['email'],
            phone=data['phone'],
        )

    @classmethod
    def from_csv_line(cls, line: str) -> 'Contact':
        """Build a Contact from a CSV line: 'Alice,alice@example.com,555-1234'"""
        parts = line.strip().split(',')
        return cls(name=parts[0], email=parts[1], phone=parts[2])


# Normal constructor
c1 = Contact('Alice', 'alice@example.com', '555-1234')
print(c1)

# Alternative constructor via @classmethod
data = {'name': 'Bob', 'email': 'bob@example.com', 'phone': '555-5678'}
c2 = Contact.from_dict(data)
print(c2)

c3 = Contact.from_csv_line('Carol,carol@example.com,555-9999')
print(c3)

## 3. @staticmethod — Utility Functions Inside a Class

In [ ]:
import re

class Validator:
    """A collection of validation utilities — no instance state needed."""

    @staticmethod
    def is_valid_email(email: str) -> bool:
        return bool(re.match(r'^[\w.-]+@[\w.-]+\.\w{2,}$', email))

    @staticmethod
    def is_valid_phone(phone: str) -> bool:
        return bool(re.match(r'^\+?[\d\s\-]{7,15}$', phone))

    @staticmethod
    def normalize_name(name: str) -> str:
        return name.strip().title()  # 'alice smith' → 'Alice Smith'


# Call directly on the class — no need to create an instance
print(Validator.is_valid_email('alice@example.com'))  # True
print(Validator.is_valid_email('not-an-email'))        # False
print(Validator.normalize_name('  alice smith  '))     # Alice Smith

## 4. Quick Reference

| | Receives | Use when |
|---|---|---|
| Instance method | `self` | Need to read/modify instance data |
| `@classmethod` | `cls` | Alternative constructors, factory methods |
| `@staticmethod` | nothing | Utility logic related to the class but not the instance |

## Exercise
1. Create a `Date` class with `from_string('2024-01-15')` as a `@classmethod` constructor.
2. Add a `@staticmethod` called `is_leap_year(year)` that returns True/False.
3. Create a `User` class with `from_dict` and `from_json_string` classmethods.